<a href="https://colab.research.google.com/github/sanskrutilanjewar83-del/Html-code-/blob/main/PROJECT_Weather_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np, pandas as pd, json, warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, mean_absolute_error,
                             mean_squared_error, r2_score)



In [3]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
N = 3652
dates = pd.date_range("2016-01-01", periods=N, freq="D")
doy = dates.dayofyear.values
ang = 2 * np.pi * (doy - 1) / 365.25

In [4]:
# ---- seasonal climatology (Nagpur-like: hot summer, monsoon Jun-Sep, mild winter)
tmax_base = 33.5 + 8.0 * np.sin(ang - 1.55) + 0.0
tmax_base = 34 + 7.5 * np.sin(ang - 1.75)
tmin_base = 20.5 + 8.0 * np.sin(ang - 1.95)
hum_base = 55 + 22 * np.sin(ang - 3.55 + np.pi) * -1
hum_base = 52 + 25 * np.clip(np.sin(ang - 2.0), -1, 1)
monsoon = np.exp(-((doy - 205) / 38.0) ** 2)



In [5]:
# ---- latent weather "wetness" state (AR process) drives clouds, humidity, pressure, rain
w = np.zeros(N)
for i in range(1, N):
    w[i] = 0.72 * w[i - 1] + rng.normal(0, 1)
wet = w + 1.6 * monsoon * 2 - 0.4



In [6]:
cloud = np.clip(35 + 18 * wet + 25 * monsoon + rng.normal(0, 9, N), 0, 100)
humidity = np.clip(hum_base + 6 * wet + rng.normal(0, 5, N), 18, 99)
pressure = 1008 + 4.5 * np.cos(ang) - 2.2 * wet - 3 * monsoon + rng.normal(0, 1.2, N)
wind = np.clip(8 + 4 * monsoon + 1.5 * np.abs(wet) + rng.normal(0, 2.5, N), 1, 40)
tmax = tmax_base - 1.6 * wet - 4.5 * monsoon + rng.normal(0, 1.4, N)
tmin = tmin_base + 0.6 * wet - 0.6 * monsoon + rng.normal(0, 1.1, N)
tmin = np.minimum(tmin, tmax - 4)


In [7]:
# ---- rainfall today depends on wetness / humidity / season (stochastic)
p_rain_today = 1 / (1 + np.exp(-(1.35 * wet + 0.035 * (humidity - 65) - 2.9 + 2.2 * monsoon)))
rain_today_flag = rng.random(N) < p_rain_today
rain_mm = np.where(rain_today_flag, rng.gamma(1.4, 6.5, N) * (0.6 + 0.8 * monsoon), 0.0)
rain_mm = np.round(rain_mm, 1)

df = pd.DataFrame(dict(Date=dates, MaxTemp=tmax.round(1), MinTemp=tmin.round(1),
                       Humidity=humidity.round(0), Pressure=pressure.round(1),
                       WindSpeed=wind.round(1), CloudCover=cloud.round(0), Rainfall=rain_mm))

# inject realistic missing values & outliers (to demonstrate preprocessing)
for c, frac in [("Humidity", .012), ("Pressure", .010), ("WindSpeed", .015), ("CloudCover", .010), ("MaxTemp", .006)]:
    idx = rng.choice(N, int(frac * N), replace=False)
    df.loc[idx, c] = np.nan
out_idx = rng.choice(N, 6, replace=False)
df.loc[out_idx, "WindSpeed"] = rng.uniform(70, 95, 6).round(1)
raw_missing = df.isna().sum().to_dict()
df.to_csv("./weather_dataset_raw.csv", index=False)



In [8]:
# =============== PREPROCESSING ===============
d = df.copy()
n_outliers = int((d.WindSpeed > 60).sum())
d.loc[d.WindSpeed > 60, "WindSpeed"] = np.nan
d = d.set_index("Date")
d = d.interpolate(method="time").bfill().ffill()

In [10]:
# =============== FEATURE ENGINEERING ===============
d["Month"] = d.index.month
d["Month_sin"] = np.sin(2 * np.pi * d.Month / 12)
d["Month_cos"] = np.cos(2 * np.pi * d.Month / 12)
d["TempRange"] = d.MaxTemp - d.MinTemp
d["PressureChange"] = d.Pressure.diff().fillna(0)
d["HumidityChange"] = d.Humidity.diff().fillna(0)
d["Rain_lag1"] = d.Rainfall.shift(1)
d["Rain_lag2"] = d.Rainfall.shift(2)
d["Rain_3d_sum"] = d.Rainfall.rolling(3).sum()
d["Humidity_3d_mean"] = d.Humidity.rolling(3).mean()
d["MaxTemp_lag1"] = d.MaxTemp.shift(1)
d["RainToday"] = (d.Rainfall >= 2.5).astype(int)          # IMD: >=2.5 mm = rainy day
d["RainTomorrow"] = d.RainToday.shift(-1)
d["MaxTemp_Tomorrow"] = d.MaxTemp.shift(-1)
d = d.dropna()
d["RainTomorrow"] = d.RainTomorrow.astype(int)

feats = ["MaxTemp", "MinTemp", "Humidity", "Pressure", "WindSpeed", "CloudCover", "Rainfall",
         "Month_sin", "Month_cos", "TempRange", "PressureChange", "HumidityChange",
         "Rain_lag1", "Rain_lag2", "Rain_3d_sum", "Humidity_3d_mean", "MaxTemp_lag1", "RainToday"]

/tmp/ipykernel_1837/1559830958.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  d["RainTomorrow"] = d.RainTomorrow.astype(int)


In [12]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, mean_absolute_error,
                             mean_squared_error, r2_score)
import json
import pickle

# Define the features and targets
X = d[feats]
y = d["RainTomorrow"]

# chronological-aware? use stratified random split as per course template (70/15/15)
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_te, y_val, y_te = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=42, stratify=y_tmp)

models = {
    "Logistic Regression": Pipeline([("s", StandardScaler()), ("m", LogisticRegression(max_iter=1000, random_state=42))]),
    "KNN": Pipeline([("s", StandardScaler()), ("m", KNeighborsClassifier(n_neighbors=11))]),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, min_samples_leaf=10, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=3, random_state=42, n_jobs=2),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=250, learning_rate=0.06, max_depth=3, random_state=42),
}
skf = StratifiedKFold(5, shuffle=True, random_state=42)
res = {}; preds = {}; probs = {}
for name, m in models.items():
    cv = cross_val_score(m, X_tr, y_tr, cv=skf, scoring="accuracy")
    m.fit(X_tr, y_tr)
    pv = m.predict(X_val)
    p = m.predict(X_te); pr = m.predict_proba(X_te)[:, 1]
    preds[name] = p; probs[name] = pr
    res[name] = dict(cv_mean=cv.mean(), val_acc=accuracy_score(y_val, pv),
                     acc=accuracy_score(y_te, p), prec=precision_score(y_te, p),
                     rec=recall_score(y_te, p), f1=f1_score(y_te, p),
                     auc=roc_auc_score(y_te, pr), cm=confusion_matrix(y_te, p).tolist())
best = max(res, key=lambda k: res[k]["f1"])

# regression: tomorrow's max temperature
yr = d["MaxTemp_Tomorrow"]
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(X, yr, test_size=0.15, random_state=42)
lr = LinearRegression().fit(Xr_tr, yr_tr)
rfr = RandomForestRegressor(n_estimators=300, max_depth=12, min_samples_leaf=3, random_state=42, n_jobs=2).fit(Xr_tr, yr_tr)
reg = {}
for nm, mm in [("Linear Regression", lr), ("Random Forest Regressor", rfr)]:
    p = mm.predict(Xr_te)
    reg[nm] = dict(mae=mean_absolute_error(yr_te, p), rmse=mean_squared_error(yr_te, p) ** .5, r2=r2_score(yr_te, p))
reg_pred = rfr.predict(Xr_te)

imp = pd.Series(models["Random Forest"].feature_importances_, index=feats).sort_values(ascending=False)

info = dict(n_raw=len(df), n_final=len(d), n_feat=len(feats), raw_missing=raw_missing, n_outliers=n_outliers,
            n_tr=len(X_tr), n_val=len(X_val), n_te=len(X_te),
            rain_pct=float(y.mean() * 100), best=best, res=res, reg=reg,
            imp=imp.head(10).to_dict(), start=str(df.Date.min().date()), end=str(df.Date.max().date()))
json.dump(info, open("./results.json", "w"), indent=1, default=float)
d.to_csv("./weather_dataset_processed.csv")

import pickle
pickle.dump(dict(d=d, df=df, y_te=y_te, preds=preds, probs=probs, imp=imp, yr_te=yr_te, reg_pred=reg_pred,
                 feats=feats, res=res, best=best), open("./state.pkl", "wb"))
for k, v in res.items():
    print(f"{k:20s} cv={v["cv_mean"]:.3f} val={v["val_acc"]:.3f} acc={v["acc"]:.3f} P={v["prec"]:.3f} R={v["rec"]:.3f} F1={v["f1"]:.3f} AUC={v["auc"]:.3f}")
print("best", best, "| rain%", round(info["rain_pct"], 1), "| reg", reg, "| missing", raw_missing, "| outl", n_outliers, "| splits", len(X_tr), len(X_val), len(X_te))
print(imp.head(8))

Logistic Regression  cv=0.887 val=0.892 acc=0.894 P=0.739 R=0.667 F1=0.701 AUC=0.919
KNN                  cv=0.889 val=0.876 acc=0.881 P=0.687 R=0.667 F1=0.677 AUC=0.913
Decision Tree        cv=0.872 val=0.872 acc=0.870 P=0.660 R=0.627 F1=0.643 AUC=0.899
Random Forest        cv=0.892 val=0.879 acc=0.891 P=0.706 R=0.706 F1=0.706 AUC=0.920
Gradient Boosting    cv=0.879 val=0.859 acc=0.876 P=0.667 R=0.667 F1=0.667 AUC=0.914
best Random Forest | rain% 18.5 | reg {'Linear Regression': {'mae': 1.8359473930188233, 'rmse': 2.304593983149733, 'r2': 0.7574028932700291}, 'Random Forest Regressor': {'mae': 1.8176710727351229, 'rmse': 2.2779696901126907, 'r2': 0.7629758203744176}} | missing {'Date': 0, 'MaxTemp': 21, 'MinTemp': 0, 'Humidity': 43, 'Pressure': 36, 'WindSpeed': 54, 'CloudCover': 36, 'Rainfall': 0} | outl 6 | splits 2554 547 548
Pressure            0.172826
Humidity            0.146327
Humidity_3d_mean    0.120181
CloudCover          0.098202
TempRange           0.076679
Rain_3d_sum   